# DLT Expectations & Data Quality

> **Note:** DLT pipeline code (`import dlt`) must run inside a DLT pipeline. Code cells in this notebook show the patterns. The Event Log query section at the end runs against a simulated event log and is executable in any Databricks notebook.

Notebook 12 introduced the three expectation modes. This notebook goes deeper: multiple expectations on one table, complex conditions, the DLT quarantine pattern, and querying the Event Log to build data quality dashboards.

In this notebook we'll cover:
- Stacking multiple expectations and how they combine
- Writing complex expectation conditions
- The DLT quarantine pattern — routing bad rows to a separate table
- Event Log structure — what gets recorded and where to find it
- Querying the Event Log for quality metrics

## Multiple Expectations on One Table

You can stack any number of expectations. Each constraint is evaluated **independently** on every row.

```python
@dlt.table(name="silver_orders")
@dlt.expect("warn_high_amount",        "amount < 10000")           # warn but keep
@dlt.expect_or_drop("positive_amount",  "amount > 0")               # drop if ≤ 0
@dlt.expect_or_drop("valid_customer",   "customer_id IS NOT NULL")  # drop if null
@dlt.expect_or_fail("valid_order_id",   "order_id IS NOT NULL")     # fail pipeline
def silver_orders():
    return dlt.read_stream("bronze_orders")
```

SQL equivalent — multiple CONSTRAINT clauses:

```sql
CREATE OR REFRESH STREAMING TABLE silver_orders
  (
    CONSTRAINT warn_high_amount  EXPECT (amount < 10000),
    CONSTRAINT positive_amount   EXPECT (amount > 0)              ON VIOLATION DROP ROW,
    CONSTRAINT valid_customer    EXPECT (customer_id IS NOT NULL) ON VIOLATION DROP ROW,
    CONSTRAINT valid_order_id    EXPECT (order_id IS NOT NULL)    ON VIOLATION FAIL UPDATE
  )
AS SELECT * FROM STREAM(LIVE.bronze_orders);
```

### How Multiple Expectations Interact

Each expectation is evaluated independently:

- A row that fails `positive_amount` is **dropped** — the `warn_high_amount` check doesn't matter for that row
- A row that fails `valid_order_id` **fails the pipeline immediately** — even if the row also fails a drop constraint
- A row that fails `warn_high_amount` but passes everything else is **written with a warning recorded**
- The fail constraint (`valid_order_id`) is checked **before** the pipeline processes any rows — if it fires, the entire micro-batch is rejected

> **Exam tip:** `expect_or_fail` is the most severe — it stops the **entire pipeline update**, not just the current row or batch. Use it only for constraints where continuing with bad data would corrupt downstream consumers.

## Complex Expectation Conditions

Expectation conditions are SQL boolean expressions — any SQL function, operator, or sub-expression is valid.

### Multi-Column Conditions

```sql
-- Business rule: discount cannot exceed amount
CONSTRAINT valid_discount EXPECT (discount <= amount)

-- End date must be after start date
CONSTRAINT valid_date_range EXPECT (end_date > start_date OR end_date IS NULL)
```

### Using SQL Functions

```sql
-- Email format (basic regex)
CONSTRAINT valid_email EXPECT (email RLIKE '^[^@]+@[^@]+\\.[^@]+$')

-- Only allow known regions
CONSTRAINT known_region EXPECT (region IN ('UK', 'US', 'DE', 'FR', 'JP'))

-- Amount must be a reasonable value (not likely a test record)
CONSTRAINT reasonable_amount EXPECT (amount BETWEEN 0.01 AND 100000)

-- Order date not in the future
CONSTRAINT not_future_date EXPECT (order_date <= CURRENT_DATE())

-- ID must be exactly 6 characters
CONSTRAINT valid_id_length EXPECT (LENGTH(order_id) = 6)
```

### Python with Complex Expressions

```python
@dlt.table(name="silver_orders")
@dlt.expect_or_drop("valid_email",
    "email IS NULL OR email RLIKE '^[^@]+@[^@]+\\.[^@]+$'")
@dlt.expect_or_drop("known_region",
    "region IN ('UK', 'US', 'DE', 'FR', 'JP')")
@dlt.expect("not_future_date",
    "order_date <= CURRENT_DATE()")
def silver_orders():
    return dlt.read_stream("bronze_orders")
```

> Null-safety matters: `email IS NULL OR email RLIKE ...` passes null emails through (to handle optionally). `email RLIKE ...` alone would fail null emails.

## The DLT Quarantine Pattern

The manual quarantine pattern (notebook 06) routes bad rows to a separate table using Python logic. DLT offers a declarative equivalent using a combination of `expect` (keep but flag) and a downstream filter.

### Pattern: Flag in Silver, Separate in Gold

```python
# Step 1: Land everything — use 'expect' to track violations WITHOUT dropping
@dlt.table(name="silver_orders_raw")
@dlt.expect("positive_amount",  "amount > 0")
@dlt.expect("valid_customer",   "customer_id IS NOT NULL")
@dlt.expect("known_region",     "region IN ('UK','US','DE','FR')")
def silver_orders_raw():
    return dlt.read_stream("bronze_orders")


# Step 2: Valid rows → clean Silver table
@dlt.table(name="silver_orders")
def silver_orders():
    return (
        dlt.read_stream("silver_orders_raw")
           .filter("amount > 0 AND customer_id IS NOT NULL AND region IN ('UK','US','DE','FR')")
    )


# Step 3: Invalid rows → quarantine table
@dlt.table(name="quarantine_orders",
           comment="Rows that failed one or more quality checks")
def quarantine_orders():
    return (
        dlt.read_stream("silver_orders_raw")
           .filter("""
             amount <= 0
             OR customer_id IS NULL
             OR region NOT IN ('UK','US','DE','FR')
           """)
    )
```

**Why this over `expect_or_drop`:**
- Bad rows are preserved in the quarantine table — you can investigate and reprocess them
- `expect_or_drop` silently removes rows — they are gone
- Expectation metrics in the event log still count the violations (from step 1)

> This is the most robust quality pattern in DLT. Combine it with `expect_or_fail` on critical constraints (like `order_id IS NOT NULL`) so the pipeline stops if truly unrecoverable data arrives.

## Event Log Structure

DLT writes every pipeline event to a Delta table — the **Event Log** — at the pipeline's configured storage location:

```
dbfs:/pipelines/<pipeline-id>/system/events
```

You query it like any Delta table:

```sql
SELECT * FROM delta.`dbfs:/pipelines/<pipeline-id>/system/events`
ORDER BY timestamp DESC LIMIT 50;
```

### Key Columns

| Column | Type | Description |
|--------|------|-------------|
| `id` | STRING | Unique event ID |
| `timestamp` | TIMESTAMP | When the event occurred |
| `origin.pipeline_id` | STRING | Which pipeline |
| `origin.flow_name` | STRING | Which table ("flow") |
| `event_type` | STRING | Type of event |
| `details` | STRING (JSON) | Event-specific payload — parse with `:` operator |
| `level` | STRING | `INFO`, `WARN`, `ERROR`, `METRICS` |
| `error` | STRUCT | Error info if event is a failure |

### Key Event Types

| `event_type` | Meaning |
|---|---|
| `create_update` | A pipeline run was triggered |
| `flow_progress` | A table started, completed, or produced metrics |
| `dataset_definition` | Schema of a declared table |
| `planning_information` | Which tables will be computed this run |
| `flow_definition` | Source query of a table |

## Querying Expectation Metrics from the Event Log

Expectation results are embedded in `flow_progress` events in the `details` JSON column. Use the `:` (colon path) operator to extract nested JSON fields.

```sql
-- Extract expectation metrics per table per run
SELECT
  timestamp,
  origin.flow_name                                  AS table_name,
  details:flow_progress.metrics.num_output_rows     AS rows_written,
  details:flow_progress.data_quality.dropped_records AS rows_dropped,
  details:flow_progress.data_quality.expectations   AS expectation_details
FROM delta.`dbfs:/pipelines/<pipeline-id>/system/events`
WHERE event_type = 'flow_progress'
  AND details:flow_progress.status = 'COMPLETED'
ORDER BY timestamp DESC;
```

The `expectations` field is an array of objects, one per constraint:

```json
[
  {"name": "positive_amount",  "dataset": "silver_orders", "passed_records": 980, "failed_records": 20},
  {"name": "valid_customer",   "dataset": "silver_orders", "passed_records": 995, "failed_records":  5}
]
```

To unnest the expectations array into rows:

```sql
SELECT
  timestamp,
  origin.flow_name                          AS table_name,
  expectation.name                          AS constraint_name,
  expectation.passed_records                AS passed,
  expectation.failed_records                AS failed,
  ROUND(100.0 * expectation.failed_records
    / (expectation.passed_records + expectation.failed_records), 2) AS failure_pct
FROM delta.`dbfs:/pipelines/<pipeline-id>/system/events`
LATERAL VIEW EXPLODE(
  FROM_JSON(
    details:flow_progress.data_quality.expectations,
    'ARRAY<STRUCT<name:STRING, passed_records:LONG, failed_records:LONG>>'
  )
) AS expectation
WHERE event_type = 'flow_progress'
ORDER BY timestamp DESC, failed DESC;
```

In [ ]:
# Simulate the Event Log structure and demonstrate the query pattern
# (In a real pipeline, query delta.`dbfs:/pipelines/<id>/system/events` instead)

import json

# Simulated flow_progress event payloads
events = [
    {
        "id":        "evt-001",
        "timestamp": "2024-01-15 09:05:00",
        "event_type":"flow_progress",
        "level":     "METRICS",
        "flow_name": "silver_orders",
        "details": json.dumps({
            "flow_progress": {
                "status": "COMPLETED",
                "metrics": {"num_output_rows": 980},
                "data_quality": {
                    "dropped_records": 15,
                    "expectations": [
                        {"name": "positive_amount", "passed_records": 990, "failed_records": 10},
                        {"name": "valid_customer",  "passed_records": 995, "failed_records":  5},
                    ]
                }
            }
        })
    },
    {
        "id":        "evt-002",
        "timestamp": "2024-01-16 09:05:00",
        "event_type":"flow_progress",
        "level":     "METRICS",
        "flow_name": "silver_orders",
        "details": json.dumps({
            "flow_progress": {
                "status": "COMPLETED",
                "metrics": {"num_output_rows": 1450},
                "data_quality": {
                    "dropped_records": 3,
                    "expectations": [
                        {"name": "positive_amount", "passed_records": 1452, "failed_records": 1},
                        {"name": "valid_customer",  "passed_records": 1451, "failed_records": 2},
                    ]
                }
            }
        })
    },
]

event_log_df = spark.createDataFrame(events)
event_log_df.createOrReplaceTempView("simulated_event_log")
print("Simulated event log created")

In [ ]:
# Query 1: Per-run summary — rows written and dropped per table
spark.sql("""
  SELECT
    timestamp,
    flow_name                                                          AS table_name,
    details:flow_progress.metrics.num_output_rows                      AS rows_written,
    details:flow_progress.data_quality.dropped_records                 AS rows_dropped
  FROM simulated_event_log
  WHERE event_type = 'flow_progress'
  ORDER BY timestamp
""").show()

In [ ]:
from pyspark.sql.functions import from_json, explode, col, schema_of_json, round as spark_round
from pyspark.sql.types import ArrayType, StructType, StructField, StringType, LongType

# Query 2: Expectation-level breakdown — failure rate per constraint per run
expectation_schema = ArrayType(StructType([
    StructField("name",            StringType(), True),
    StructField("passed_records",  LongType(),   True),
    StructField("failed_records",  LongType(),   True),
]))

(
    spark.table("simulated_event_log")
    .filter("event_type = 'flow_progress'")
    .withColumn(
        "expectations",
        from_json(
            col("details")["flow_progress"]["data_quality"]["expectations"],
            expectation_schema
        )
    )
    .withColumn("exp", explode("expectations"))
    .select(
        col("timestamp"),
        col("flow_name").alias("table_name"),
        col("exp.name").alias("constraint"),
        col("exp.passed_records").alias("passed"),
        col("exp.failed_records").alias("failed"),
        spark_round(
            100.0 * col("exp.failed_records") /
            (col("exp.passed_records") + col("exp.failed_records")), 2
        ).alias("failure_pct")
    )
    .orderBy("timestamp", "constraint")
    .show(truncate=False)
)

## Choosing the Right Failure Mode — Decision Guide

```
Question                                             → Mode
────────────────────────────────────────────────────────────────────────────
The data is still useful even with this issue?       → expect (warn)
Bad rows should not reach Silver, but pipeline ok?   → expect_or_drop
Continuing with this violation would corrupt data?   → expect_or_fail
I want bad rows in a separate table to investigate?  → expect (warn) + downstream filter
```

### Naming Conventions

Good expectation names make the Event Log readable:

```python
# Describe what SHOULD be true (not what is wrong)
@dlt.expect_or_drop("amount_is_positive",    "amount > 0")          # ✓
@dlt.expect_or_drop("amount_not_negative",   "amount >= 0")         # ✓
@dlt.expect_or_drop("not_negative_amount",   "amount > 0")          # ✗ less clear
@dlt.expect_or_drop("bad_amount",            "amount > 0")          # ✗ confusing — name says 'bad' but fires on bad data
```

### What to Expect in the Exam

| Scenario | Answer |
|----------|--------|
| Pipeline should continue even if some rows have null emails | `expect` (warn) |
| Rows with negative amounts must not reach Silver | `expect_or_drop` |
| A missing primary key means all downstream joins are broken | `expect_or_fail` |
| You want to investigate rejected rows later | quarantine pattern (`expect` + downstream filter) |
| You want quality metrics but don't want to stop or drop | `expect` only |
| New column in source should not break the pipeline | `cloudFiles.schemaEvolutionMode = rescue` (Auto Loader, not expectations) |

In [ ]:
# Cleanup
spark.catalog.dropTempView("simulated_event_log")

## Summary

- Stack multiple expectations on one table — each is evaluated independently. `expect_or_fail` stops the entire pipeline update, so use it sparingly for truly unrecoverable violations.
- Expectation conditions are full SQL boolean expressions — use `RLIKE`, `IN`, `BETWEEN`, `LENGTH`, `CURRENT_DATE()`, multi-column expressions, and null-safe patterns (`col IS NULL OR col RLIKE ...`).
- The **DLT quarantine pattern**: use `expect` (warn, keep rows) on an intermediate table, then filter valid rows to Silver and invalid rows to a quarantine table. Preserves bad rows for investigation unlike `expect_or_drop`.
- The **Event Log** is a queryable Delta table at `dbfs:/pipelines/<pipeline-id>/system/events`. Key event type for quality: `flow_progress` with `level = 'METRICS'`.
- Expectation metrics are in `details:flow_progress.data_quality.expectations` as a JSON array — use `FROM_JSON` + `EXPLODE` (or the `:` path operator) to extract per-constraint pass/fail counts.
- Naming: describe what **should be true**, not what is wrong. Good names make the Event Log readable.
- Decision: `expect` = warn and keep; `expect_or_drop` = remove bad rows; `expect_or_fail` = stop pipeline; quarantine pattern = keep bad rows for reprocessing.

Next: `14-databricks-workflows-jobs.ipynb` — orchestrating pipelines with Databricks Jobs: tasks, dependencies, triggers, and alerts.